In [41]:
# Imports
import os, re, pathlib
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from kmeans_model import KMeansTF, save_model
from scipy.signal import welch

In [42]:
# Configuration
data_root = r"/home/elizabeth/Documents/Data_clean/Data_clean"
skip_dirs = { 'Group1-8channels' }
# PCA: False = use all features (often better); True = reduce dimension (faster, sometimes worse)
use_pca = True
pca_components = 32  # only used if use_pca True; try 64 or None for more variance
n_clusters = 60  # Number of clusters is multiple for the 6 categories
max_iter = 5000  # More iterations can help convergence on large data
tol = 1e-6  # Convergence tolerance
random_state = 42  # Try 0, 1, 42, 123 and pick best accuracy if desired

In [43]:
# Load all EEG files recursively (OpenBCI: % comment lines, then header, then data)
# Also detect per-file sampling rate (e.g., 125 Hz vs 250 Hz) from the header
core_dir = pathlib.Path(data_root)
dfs = []
fs_values = []

for item in core_dir.rglob('*.txt'):
    try:
        if set(item.parts).isdisjoint(skip_dirs):
            # Detect sampling rate from commented header lines (e.g. "Sample Rate = 250.0 Hz")
            file_fs = 125.0  # sensible default if we cannot parse
            try:
                with open(item, 'r', errors='ignore') as f:
                    for line in f:
                        if not line.startswith('%'):
                            break
                        m = re.search(r'(\d+(\.\d+)?)\s*Hz', line)
                        if m:
                            file_fs = float(m.group(1))
                            break
            except Exception as e_head:
                print(f'Warning: could not parse sample rate from header in {item}: {e_head}')

            # Skip lines starting with %; first non-comment line is the header
            df = pd.read_csv(item, sep=',', comment='%', on_bad_lines='skip')
            if df.empty or df.shape[1] < 2:
                continue
            df['src_filename'] = str(item)
            df['fs'] = file_fs
            dfs.append(df)
            fs_values.append(file_fs)
    except Exception as e:
        print(f'Failed to read {item}: {e}')

print(f'Read {len(dfs)} files')
eeg_data = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(eeg_data.shape)
if fs_values:
    fs_series = pd.Series(fs_values)
    print('Per-file sample rates (Hz):')
    print(fs_series.value_counts().sort_index())

# (1.5 min time taken)

Failed to read /home/elizabeth/Documents/Data_clean/Data_clean/G04/person09/Backward/OpenBCI-RAW-2025-03-14_15-56-17.txt: No columns to parse from file
Read 1816 files
(2885310, 35)
Per-file sample rates (Hz):
125.0    1394
250.0     422
Name: count, dtype: int64


In [44]:
# Robust label normalization from filenames 
assert not eeg_data.empty, 'No data loaded. Check data_root.'
src = eeg_data['src_filename'].astype(str).str.lower()
src_norm = src.str.replace(r'[\s_\-]+', '', regex=True)
labels = pd.Series('', index=eeg_data.index)
def assign_where(patterns, value, labels, src_norm):
    m = pd.Series(False, index=src_norm.index)
    for pat in patterns:
        m = m | src_norm.str.contains(pat)
    cond = (labels == '') & m
    return labels.where(~cond, other=value)
labels = assign_where(['backward','backwards','backard'], 'backward', labels, src_norm)
labels = assign_where(['fowward','forward','foreward'], 'forward', labels, src_norm)
labels = assign_where(['landing'], 'landing', labels, src_norm)
labels = assign_where(['left'], 'left', labels, src_norm)
labels = assign_where(['right'], 'right', labels, src_norm)
labels = assign_where(['takeoff','takeoff','takoff'], 'takeoff', labels, src_norm)
eeg_data['label_txt'] = labels.astype(str)
print(eeg_data.groupby(['label_txt'])['src_filename'].count())
# Temporary settings to show the full path - use if label_txt shows having a value
# with pd.option_context('display.max_colwidth', None):
#     print(eeg_data[eeg_data['label_txt'] == '']['src_filename'].head(20))
#(2.5 min to run)

label_txt
backward    478451
forward     485550
landing     479132
left        473995
right       481853
takeoff     486329
Name: src_filename, dtype: int64


In [45]:
# Build feature matrix X and label vector y
df = eeg_data.copy()
df = df[df['label_txt'].astype(str).str.len() > 0]

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Updated to drop ALL non-EEG columns from the feature set
# Updated to KEEP Accelerometer data
drop_keywords = ['Sample', 'Timestamp', 'Marker', 'Not Used', 'Analog', 'Digital', 'Other']
drop_cols = [c for c in num_cols if any(key in c for key in drop_keywords)]

# Never treat the sampling-rate column as an EEG feature
if 'fs' in num_cols:
    drop_cols.append('fs')

drop_cols = sorted(set(drop_cols))
feature_cols = [c for c in num_cols if c not in drop_cols]

feat = df[feature_cols].copy()
feat = feat.replace([np.inf, -np.inf], np.nan)
feat = feat.dropna(axis=1, how='all')
med = feat.median(numeric_only=True)
feat = feat.fillna(med)
std = feat.std(numeric_only=True)
keep = std[std > 0].index.tolist()
if len(keep) == 0:
    # Fallback: use all numeric columns (your files may use different sep/header)
    keep = feature_cols
    print("Warning: no columns had std > 0; using all numeric columns. If PCA/K-means fails, check CSV format: try sep='\\t', header=None in the load cell.")

feat = feat[keep]
X = feat.to_numpy(dtype=np.float32)

y_cat = df['label_txt'].astype('category')
y = y_cat.cat.codes.to_numpy(dtype=np.int32)
label_names = list(y_cat.cat.categories)

# Store numeric labels back into df so we can use them during PSD windowing
df['y_code'] = y

print('X shape:', X.shape, 'classes:', len(label_names))
print(f"Features being used for K-Means: {keep}")  # shows the channels we are keeping
# (1 min to run)

X shape: (2885310, 19) classes: 6
Features being used for K-Means: [' EXG Channel 0', ' EXG Channel 1', ' EXG Channel 2', ' EXG Channel 3', ' EXG Channel 4', ' EXG Channel 5', ' EXG Channel 6', ' EXG Channel 7', ' EXG Channel 8', ' EXG Channel 9', ' EXG Channel 10', ' EXG Channel 11', ' EXG Channel 12', ' EXG Channel 13', ' EXG Channel 14', ' EXG Channel 15', ' Accel Channel 0', ' Accel Channel 1', ' Accel Channel 2']


In [46]:

# --- 1. EXTRACT POWER FEATURES WITH PER-FILE SAMPLING RATE ---
# We convert raw voltage into Alpha/Beta "Volume", respecting 125 Hz vs 250 Hz, etc.
X_psd = []
y_psd = []
psd_feature_names = []

# Part A = EEG only (theta/alpha/beta); Part B = accel mean (tilt). Don't run Welch on accel.
eeg_idx = [i for i, c in enumerate(keep) if 'Accel' not in c]
accel_idx = [i for i, c in enumerate(keep) if 'Accel' in c]
n_eeg = len(eeg_idx)
n_accel = len(accel_idx)

# Feature names: brainwave bands per EEG channel, then accel (tilt)
for ch in range(n_eeg):
    psd_feature_names.extend([f'ch{ch}_theta', f'ch{ch}_alpha', f'ch{ch}_beta'])
for j in range(n_accel):
    psd_feature_names.append(f'accel_{j}')

print("Converting raw voltage to Delta/Theta/Alpha/Beta/Gamma Power (Frequency) with per-file sample rates...")

# Work per file so we don't mix different sampling rates in a single window
required_cols = keep + ['y_code', 'fs', 'src_filename']
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for PSD extraction: {missing_cols}")

# Ensure we only use the columns we care about
work_df = df[required_cols].copy()

for fname, g in work_df.groupby('src_filename', sort=False):
    file_fs = float(g['fs'].iloc[0]) if 'fs' in g.columns else 125.0
    if file_fs <= 0:
        file_fs = 125.0  # fallback
    win_size = int(file_fs)  # 1-second window in samples

    X_file = g[keep].to_numpy(dtype=np.float32)
    y_file = g['y_code'].to_numpy(dtype=np.int32)

    if X_file.shape[0] < win_size:
        # Too short to form even a single 1-second window
        continue

    for start in range(0, X_file.shape[0] - win_size, win_size):
        window = X_file[start:start + win_size, :]
        row_feats = []

        # --- Part A: Brainwaves (EEG only; Welch delta/theta/alpha/beta/gamma) ---
        for ch_idx in eeg_idx:
            freqs, psd = welch(window[:, ch_idx], file_fs, nperseg=win_size)
            delta = np.mean(psd[(freqs >= 0.5) & (freqs < 4)])
            theta = np.mean(psd[(freqs >= 4) & (freqs <= 8)])
            alpha = np.mean(psd[(freqs >= 8) & (freqs <= 13)])
            beta = np.mean(psd[(freqs >= 13) & (freqs <= 30)])
            gamma = np.mean(psd[(freqs >= 30) & (freqs < 60)])
            row_feats.extend([delta,theta, alpha, beta, gamma])

        # --- Part B: Physical tilt (mean accel over the 1s window) ---
        if n_accel > 0:
            accel_vals = np.mean(window[:, accel_idx], axis=0)
            row_feats.extend(accel_vals.tolist())

        X_psd.append(row_feats)
        y_psd.append(y_file[start])  # label at the start of the 1-second window

X_psd = np.array(X_psd, dtype=np.float32)
y_psd = np.array(y_psd, dtype=np.int32)

print(f"Built {X_psd.shape[0]} PSD windows with {X_psd.shape[1]} features each.")

# --- 2. TRAIN/TEST SPLIT ---
X_train, X_test, y_train, y_test = train_test_split(
    X_psd, y_psd, test_size=0.2, random_state=42, stratify=y_psd
)

# --- 3. RE-SCALING (Z-Score; log PSD only) ---
# PSD bandpower is non-negative (good for log). Accel tilt can be negative (not valid for log10).
psd_dim = n_eeg * 5

X_train_psd = np.maximum(X_train[:, :psd_dim], 0.0)
X_test_psd = np.maximum(X_test[:, :psd_dim], 0.0)

X_train_psd_log = np.log10(X_train_psd + 1e-13)
X_test_psd_log = np.log10(X_test_psd + 1e-13)

if n_accel > 0:
    X_train_proc = np.concatenate([X_train_psd_log, X_train[:, psd_dim:]], axis=1)
    X_test_proc = np.concatenate([X_test_psd_log, X_test[:, psd_dim:]], axis=1)
else:
    X_train_proc = X_train_psd_log
    X_test_proc = X_test_psd_log

mu = np.nanmean(X_train_proc, axis=0)
sd = np.nanstd(X_train_proc, axis=0)

# If any feature is "dead" (sd=0), replace it so we don't get NaN
sd[sd == 0] = 1.0
mu[np.isnan(mu)] = 0.0

X_train_z = (X_train_proc - mu) / sd
X_test_z = (X_test_proc - mu) / sd

# Final check: remove any leftover NaNs
X_train_z = np.nan_to_num(X_train_z)
X_test_z = np.nan_to_num(X_test_z)

print(f"Rescaling complete. Any NaNs left? {np.isnan(X_train_z).any()}")
print(f'New Training Shape: {X_train_z.shape}')

# (Model training and evaluation are done in the next cell.)
# (12 minute to run)


Converting raw voltage to Delta/Theta/Alpha/Beta/Gamma Power (Frequency) with per-file sample rates...
Built 17889 PSD windows with 83 features each.
Rescaling complete. Any NaNs left? False
New Training Shape: (14311, 83)


In [47]:
from sklearn.decomposition import PCA

# --- PCA DIMENSIONALITY REDUCTION ---
# We use PCA to extract the 'essence' of the 85 features
# 15-20 components is usually the 'sweet spot' for EEG
n_pca = 15 
pca = PCA(n_components=n_pca, random_state=42)

X_train_pca = pca.fit_transform(X_train_z)
X_test_pca = pca.transform(X_test_z)

# Let's see how much of the 'story' we kept
explained_var = np.sum(pca.explained_variance_ratio_) * 100
print(f"PCA complete. Retained {explained_var:.2f}% of the original data variance.")
print(f"New feature count for K-Means: {X_train_pca.shape[1]}")

# --- UPDATE MODEL INPUT ---
# Now pass X_train_pca instead of X_train_z to the model
model = KMeansTF(n_clusters=n_clusters, max_iter=max_iter, tol=tol, random_state=random_state)
model.fit(X_train_pca, y_train)
y_pred = model.predict(X_test_pca)


PCA complete. Retained 93.35% of the original data variance.
New feature count for K-Means: 15


In [48]:
# # --- INITIALIZE & TRAIN ---

# model = KMeansTF(n_clusters=n_clusters, max_iter=max_iter, tol=tol, random_state=random_state)

# print("Starting KMeansTF training on Frequency Features...")
# model.fit(X_train_z, y_train)

# print(f'Inertia (tightness): {model.inertia_:.2f}')
# print('Cluster-to-label mapping:')
# for k, v in sorted(model.cluster_to_label_map_.items()):
#     print(f'  cluster {k} -> {label_names[v]} (class {v})')

# # --- PREDICT ON TEST SET ---
# print("Predicting on test set...")
# y_pred = model.predict(X_test_z)

# # --- RESULTS ---
# acc = accuracy_score(y_test, y_pred)
# print(f'\nOverall Accuracy: {acc:.4f}')

# print('\nConfusion Matrix:')
# print(confusion_matrix(y_test, y_pred))

# print('\nDetailed Performance Report:')
# # target_names uses the labels from your 'y_cat' mapping
# print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

################################################################################3
# --- INITIALIZE & TRAIN (REFINED) ---
# Use the PCA variables created in the previous step
model = KMeansTF(n_clusters=n_clusters, max_iter=max_iter, tol=tol, random_state=random_state)

print(f"Starting KMeansTF training on {X_train_pca.shape[1]} PCA components...")
model.fit(X_train_pca, y_train)

print(f'Inertia (tightness): {model.inertia_:.2f}')
print('Cluster-to-label mapping:')
for k, v in sorted(model.cluster_to_label_map_.items()):
    print(f'  cluster {k} -> {label_names[v]} (class {v})')

# --- PREDICT ON TEST SET ---
print("Predicting on test set using PCA features...")
y_pred = model.predict(X_test_pca) # Must use X_test_pca here!

# --- RESULTS ---
acc = accuracy_score(y_test, y_pred)
print(f'\nOverall Accuracy: {acc:.4f}')

print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))

print('\nDetailed Performance Report:')
print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))




Starting KMeansTF training on 15 PCA components...
Inertia (tightness): 54445.84
Cluster-to-label mapping:
  cluster 0 -> landing (class 2)
  cluster 1 -> right (class 4)
  cluster 2 -> takeoff (class 5)
  cluster 3 -> landing (class 2)
  cluster 4 -> landing (class 2)
  cluster 5 -> right (class 4)
  cluster 6 -> backward (class 0)
  cluster 7 -> takeoff (class 5)
  cluster 8 -> right (class 4)
  cluster 9 -> forward (class 1)
  cluster 10 -> forward (class 1)
  cluster 11 -> forward (class 1)
  cluster 12 -> forward (class 1)
  cluster 13 -> landing (class 2)
  cluster 14 -> forward (class 1)
  cluster 15 -> left (class 3)
  cluster 16 -> takeoff (class 5)
  cluster 17 -> forward (class 1)
  cluster 18 -> forward (class 1)
  cluster 19 -> backward (class 0)
  cluster 20 -> landing (class 2)
  cluster 21 -> left (class 3)
  cluster 22 -> backward (class 0)
  cluster 23 -> takeoff (class 5)
  cluster 24 -> forward (class 1)
  cluster 25 -> right (class 4)
  cluster 26 -> takeoff (class

In [ ]:
# Save trained model with preprocessing metadata
# Note: model is trained on PSD (alpha/beta) features, not raw EXG voltages.
meta = {
    'mu': mu.astype(np.float32),
    'sd': sd.astype(np.float32),
    'label_names': label_names,
    'feature_names': psd_feature_names,
}
out_path = 'kmeans_trained.pth'
save_model(out_path, model, meta)
print(f'Saved trained model to {out_path}')

In [ ]:
# Reload model module to ensure latest save_model is in scope
import importlib, kmeans_model
importlib.reload(kmeans_model)
from kmeans_model import KMeansTF, save_model